In [ ]:
from moccasin import setup_notebook

setup_notebook()    

In [8]:
from moccasin.config import get_active_network

active_network = get_active_network()
print(f"Active network: {active_network.name}")

Active network: eth-forked


In [ ]:
usdc = active_network.manifest_named("usdc")
weth = active_network.manifest_named("weth")
print(f"{usdc.balanceOf(boa.env.eoa)} USDC")
print(f"{weth.balanceOf(boa.env.eoa)} WETH")    

In [ ]:
usdc

In [ ]:
from moccasin.config import get_or_initialize_config
config = get_or_initialize_config()
config.reload()

active_network = config.get_active_network()
aavev3_pool_address_provider = active_network.manifest_named("aavev3_pool_address_provider")
pool_address = aavev3_pool_address_provider.getPool()
print(f"Aave V3 Pool Address: {pool_address}")

config.reload()

active_network = config.get_active_network()
pool_contract = active_network.manifest_named("pool", address=pool_address)
print(f"Aave V3 Pool Contract: {pool_contract.address}")

Aave V3 Pool Address: 0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2


In [ ]:
REFERAL_CODE = 0

def deposit(pool_contract, token, amount):    
    allowed_amount = token.allowance(boa.env.eoa, pool_contract.address)
    if allowed_amount < amount:
        token.approve(pool_contract.address, amount)
    
    print(f"Depositing {amount} of token {token.address} into Aave V3 Pool at address {pool_contract.address}")
    
    pool_contract.supply(token.address, amount, boa.env.eoa, REFERAL_CODE)

usdc_balance = usdc.balanceOf(boa.env.eoa)
weth_balance = weth.balanceOf(boa.env.eoa)
print(f"USDC Balance: {usdc_balance}")
print(f"WETH Balance: {weth_balance}")

if usdc_balance > 0:
    deposit(pool_contract, usdc, usdc_balance)

if weth_balance > 0:
    deposit(pool_contract, weth, weth_balance)
    
totalCollateralBase, totalDebtBase, availableBorrowsBase, currentLiquidationThreshold, ltv, healthFactor = pool_contract.getUserAccountData(boa.env.eoa)
print(f"Total Collateral Base: {totalCollateralBase}")
print(f"Total Debt Base: {totalDebtBase}")
print(f"Available Borrows Base: {availableBorrowsBase}")
print(f"Current Liquidation Threshold: {currentLiquidationThreshold}")
print(f"LTV: {ltv}")
print(f"Health Factor: {healthFactor}")

In [ ]:
config.reload()
active_network = config.get_active_network()
aavev3_protocol_data_provider = active_network.manifest_named("aavev3_protocol_data_provider")

a_tokens = aavev3_protocol_data_provider.getAllATokens()
print(f"Aave V3 aTokens: {a_tokens}")



In [22]:
for a_token in a_tokens:
    if "WETH" in a_token[0]:
        a_weth = active_network.manifest_named("weth", address=a_token[1])
    if "USDC" in a_token[0]:
        a_usdc = active_network.manifest_named("usdc", address=a_token[1])
        
print(f"Aave V3 aWETH Contract: {a_weth.address} {a_weth.name()}")
print(f"Aave V3 aUSDC Contract: {a_usdc.address} {a_usdc.name()}")

Aave V3 aWETH Contract: 0x4d5F47FA6A74757f35C14fD3a6Ef8E3C9BC514E8 Aave Ethereum WETH
Aave V3 aUSDC Contract: 0x98C23E9d8f34FEFb1B7BD6a91B7FF122F4e16F5c Aave Ethereum USDC


In [23]:
a_usdc_balance = a_usdc.balanceOf(boa.env.eoa)
print(f"aUSDC Balance: {a_usdc_balance}")
a_weth_balance = a_weth.balanceOf(boa.env.eoa)
print(f"aWETH Balance: {a_weth_balance}")

aUSDC Balance: 199999999
aWETH Balance: 1999999999999999999


In [24]:
a_usdc_balance_normalized = a_usdc_balance / 1e6
a_weth_balance_normalized = a_weth_balance / 1e18
print(f"aUSDC Balance Normalized: {a_usdc_balance_normalized}")
print(f"aWETH Balance Normalized: {a_weth_balance_normalized}")

aUSDC Balance Normalized: 199.999999
aWETH Balance Normalized: 2.0


In [25]:
config.reload()

def get_price(feed_name: str) -> float:
    active_network = get_active_network()
    price_feed = active_network.manifest_named(feed_name)
    price = price_feed.latestAnswer()
    decimals = price_feed.decimals()
    decimals_normalized = 10 ** decimals
    return price / decimals_normalized

usdc_price = get_price("usdc_usd")
eth_proce = get_price("eth_usd")
print(f"USDC Price: {usdc_price}")
print(f"ETH Price: {eth_proce}")

USDC Price: 0.99979
ETH Price: 2189.51089


In [26]:
usdc_value = a_usdc_balance_normalized * usdc_price
weth_value = a_weth_balance_normalized * eth_proce
total_value = usdc_value + weth_value
print(f"USDC Value: {usdc_value}")
print(f"WETH Value: {weth_value}")

target_usdc_percent_allocation = 0.3
target_weth_percent_allocation = 0.7

usdc_percent_allocation = usdc_value / total_value
weth_percent_allocation = weth_value / total_value

BUFFER = 0.1

print(f"USDC Percent Allocation: {usdc_percent_allocation}")
print(f"WETH Percent Allocation: {weth_percent_allocation}")

needs_rebalancing = abs(usdc_percent_allocation - target_usdc_percent_allocation) > BUFFER or abs(weth_percent_allocation - target_weth_percent_allocation) > BUFFER
print(f"Needs Rebalancing: {needs_rebalancing}")

USDC Value: 199.95799900020998
WETH Value: 4379.02178
USDC Percent Allocation: 0.04366867919295976
WETH Percent Allocation: 0.9563313208070402
Needs Rebalancing: True


In [27]:
a_weth.approve(pool_contract.address, a_weth_balance)
pool_contract.withdraw(weth.address, a_weth_balance, boa.env.eoa)

def print_token_balances():
    print(f"USDC Balance: {usdc.balanceOf(boa.env.eoa)}")
    print(f"WETH Balance: {weth.balanceOf(boa.env.eoa)}")   
    print(f"aUSDC Balance: {a_usdc.balanceOf(boa.env.eoa)}")   
    print(f"aWETH Balance: {a_weth.balanceOf(boa.env.eoa)}")
    
print_token_balances()

USDC Balance: 0
WETH Balance: 1999999999999999999
aUSDC Balance: 199999999
aWETH Balance: 0


In [30]:
usdc_data = { "balance": a_usdc_balance_normalized, "price": usdc_price, "contract": usdc }
weth_data = { "balance": a_weth_balance_normalized, "price": eth_proce, "contract": weth }
target_allocations = { "usdc" : 0.3, "weth": 0.7 }

def calculate_rebalancing_trades(
    usdc_data: dict,  # {"balance": float, "price": float, "contract": Contract}
    weth_data: dict,  # {"balance": float, "price": float, "contract": Contract}
    target_allocations: dict[str, float],  # {"usdc": 0.3, "weth": 0.7}
) -> dict[str, dict]:
    """
    Calculate the trades needed to rebalance a portfolio of USDC and WETH.

    Args:
        usdc_data: Dict containing USDC balance, price and contract
        weth_data: Dict containing WETH balance, price and contract
        target_allocations: Dict of token symbol to target allocation (must sum to 1)

    Returns:
        Dict of token symbol to dict containing contract and trade amount:
            {"usdc": {"contract": Contract, "trade": int},
             "weth": {"contract": Contract, "trade": int}}
    """
    # Calculate current values
    usdc_value = usdc_data["balance"] * usdc_data["price"]
    weth_value = weth_data["balance"] * weth_data["price"]
    total_value = usdc_value + weth_value

    # Calculate target values
    target_usdc_value = total_value * target_allocations["usdc"]
    target_weth_value = total_value * target_allocations["weth"]

    # Calculate trades needed in USD
    usdc_trade_usd = target_usdc_value - usdc_value
    weth_trade_usd = target_weth_value - weth_value

    # Convert to token amounts
    return {
        "usdc": {
            "contract": usdc_data["contract"],
            "trade": usdc_trade_usd / usdc_data["price"],
        },
        "weth": {
            "contract": weth_data["contract"],
            "trade": weth_trade_usd / weth_data["price"],
        },
    }

trades = calculate_rebalancing_trades(usdc_data, weth_data, target_allocations)

print(trades)

print(f"Weth to sell (negative means sell, positive means buy): {trades['weth']['trade']}")

{'usdc': {'contract': <usdc interface at 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48>, 'trade': 1173.982471018767}, 'weth': {'contract': <weth interface at 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2>, 'trade': -0.5360722068387852}}
Weth to sell (negative means sell, positive means buy): -0.5360722068387852


In [ ]:
"""
     ISwapRouter.ExactInputSingleParams memory params =
            ISwapRouter.ExactInputSingleParams({
                tokenIn: DAI,
                tokenOut: WETH9,
                fee: poolFee,
                recipient: msg.sender,
                deadline: block.timestamp,
                amountIn: amountIn,
                amountOutMinimum: 0,
                sqrtPriceLimitX96: 0
            });
            
"""

config.reload()
active_network = config.get_active_network()
uniswap_swap_router = active_network.manifest_named("uniswap_swap_router")
print(f"Uniswap Swap Router Contract: {uniswap_swap_router.address}")

weth_to_sell = int(abs(trades['weth']['trade']) * 1e18)
min_out = int(trades['usdc']['trade'] * 1e6 * 0.90)
print(f"WETH to sell (in wei): {weth_to_sell}")
weth.approve(uniswap_swap_router.address, weth_to_sell)
uniswap_swap_router.exactInputSingle((weth.address, usdc.address, int(3000), boa.env.eoa, weth_to_sell, min_out, 0))
 

Uniswap Swap Router Contract: 0x68b3465833fb72A70ecDF485E0e4C7bD8665Fc45
WETH to sell (in wei): 536072206838785216
USDC Balance: 1170060691
WETH Balance: 1463927793161214783
aUSDC Balance: 199999999
aWETH Balance: 0


In [38]:
deposit(pool_contract, usdc, usdc.balanceOf(boa.env.eoa))
deposit(pool_contract, weth, weth.balanceOf(boa.env.eoa))

print_token_balances()

Depositing 1170060691 of token 0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48 into Aave V3 Pool at address 0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2
Depositing 1463927793161214783 of token 0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2 into Aave V3 Pool at address 0x87870Bca3F3fD6335C3F4ce8392D69350B4fA4E2
USDC Balance: 0
WETH Balance: 0
aUSDC Balance: 1370060689
aWETH Balance: 1463927793161214782


In [1]:
a_usdc_balance = a_usdc.balanceOf(boa.env.eoa)
a_weth_balance = a_weth.balanceOf(boa.env.eoa)

a_usdc_balance_normalized = a_usdc_balance / 1e6
a_weth_balance_normalized = a_weth_balance / 1e18

usdc_value = a_usdc_balance_normalized * usdc_price
weth_value = a_weth_balance_normalized * eth_proce
total_value = usdc_value + weth_value

weth_percent_allocation = weth_value / total_value
print(f"New WETH Percent Allocation: {weth_percent_allocation}")
usdc_percent_allocation = usdc_value / total_value
print(f"New USDC Percent Allocation: {usdc_percent_allocation}")


NameError: name 'a_usdc' is not defined